# 09 - Incident Root Cause Analysis Agent

An agentic system that **parses log files, correlates events, and performs hypothesis-driven root cause analysis** using OpenAI function-calling.

| Component | Detail |
|-----------|--------|
| Log Data | Loghub-style Apache & HDFS logs (embedded) |
| LLM | OpenAI GPT-4o-mini via function calling |
| Pattern | Hypothesis-test loop with evidence accumulation |

## Why an Agent?

Root cause analysis (RCA) is one of the hardest operational tasks because it requires:

1. **Pattern recognition across heterogeneous logs** -- different services, different formats.
2. **Temporal reasoning** -- understanding that Event A at t=0 could cause Event B at t=5s.
3. **Hypothesis generation and testing** -- forming theories, checking them against evidence, refining.
4. **Dependency awareness** -- knowing that Service A depends on Database B which depends on Network C.

A static pipeline can grep for errors, but an agent can **reason** about causality, form hypotheses, and iterate until the root cause is identified. This mirrors how experienced SREs actually diagnose incidents.

In [ ]:
%pip install openai pandas --quiet

In [ ]:
import json
import re
import textwrap
from datetime import datetime, timedelta

import pandas as pd
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-4o-mini"

## 1 - Sample Log Data

We embed realistic log entries based on formats found in the [Loghub](https://github.com/logpai/loghub) dataset collection. These simulate a cascading failure: database connection pool exhaustion causes API timeouts, which causes frontend errors.

In [ ]:
# Simulated incident logs -- a cascading failure scenario
INCIDENT_LOGS = """
2025-11-15 14:30:01 [db-primary] INFO  Connection pool size: 95/100
2025-11-15 14:30:15 [db-primary] WARN  Connection pool size: 98/100
2025-11-15 14:30:22 [db-primary] WARN  Slow query detected: SELECT * FROM user_sessions WHERE updated_at > '2025-11-14' (3.2s)
2025-11-15 14:30:30 [db-primary] ERROR Connection pool exhausted (100/100). New connections queued.
2025-11-15 14:30:31 [db-primary] ERROR Connection pool exhausted (100/100). 5 connections queued.
2025-11-15 14:30:32 [db-primary] ERROR Connection pool exhausted (100/100). 12 connections queued.
2025-11-15 14:30:33 [api-gateway] WARN  Upstream timeout: user-service (5000ms exceeded)
2025-11-15 14:30:34 [user-service] ERROR Database connection timeout after 5000ms
2025-11-15 14:30:34 [user-service] ERROR Failed to fetch user profile: connection_timeout
2025-11-15 14:30:35 [api-gateway] ERROR 504 Gateway Timeout - GET /api/v2/users/profile
2025-11-15 14:30:35 [api-gateway] ERROR 504 Gateway Timeout - GET /api/v2/users/settings
2025-11-15 14:30:36 [frontend] ERROR Unhandled API error: 504 on /api/v2/users/profile
2025-11-15 14:30:36 [frontend] ERROR Unhandled API error: 504 on /api/v2/users/settings
2025-11-15 14:30:37 [user-service] ERROR Database connection timeout after 5000ms
2025-11-15 14:30:37 [monitoring] ALERT Error rate exceeded threshold: user-service (errors=15, threshold=5)
2025-11-15 14:30:38 [cache-service] WARN  Cache miss rate elevated: 45% (normal: 12%)
2025-11-15 14:30:39 [db-primary] ERROR Connection pool exhausted (100/100). 28 connections queued.
2025-11-15 14:30:40 [api-gateway] ERROR 504 Gateway Timeout - POST /api/v2/orders/create
2025-11-15 14:30:40 [order-service] ERROR Failed to validate user: upstream_timeout
2025-11-15 14:30:41 [monitoring] ALERT Error rate exceeded threshold: order-service (errors=8, threshold=5)
2025-11-15 14:30:42 [db-primary] WARN  Slow query detected: SELECT COUNT(*) FROM orders WHERE status='pending' (4.1s)
2025-11-15 14:30:43 [load-balancer] WARN  Backend health check failed: user-service-02 (3 consecutive failures)
2025-11-15 14:30:44 [load-balancer] INFO  Removing user-service-02 from rotation
2025-11-15 14:30:45 [db-primary] ERROR Connection pool exhausted (100/100). 41 connections queued.
2025-11-15 14:30:46 [monitoring] ALERT P1 Incident auto-created: Multiple service failures detected
2025-11-15 14:30:50 [cron-scheduler] INFO  Started: nightly_session_cleanup job
2025-11-15 14:30:51 [db-primary] WARN  Bulk DELETE on user_sessions table: 450000 rows affected
2025-11-15 14:30:55 [db-primary] INFO  Connection pool size: 72/100 (draining)
2025-11-15 14:31:10 [db-primary] INFO  Connection pool size: 45/100
2025-11-15 14:31:30 [user-service] INFO  Database connection restored
2025-11-15 14:31:35 [api-gateway] INFO  Upstream user-service responding normally (latency: 85ms)
2025-11-15 14:31:40 [monitoring] INFO  Error rates returning to normal
""".strip()

# Additional context logs from earlier in the day
CONTEXT_LOGS = """
2025-11-15 02:00:00 [cron-scheduler] INFO  Started: nightly_session_cleanup job
2025-11-15 02:00:05 [db-primary] INFO  Bulk DELETE on user_sessions table: 12000 rows affected
2025-11-15 02:00:06 [db-primary] INFO  Connection pool size: 15/100
2025-11-15 09:00:00 [deploy-bot] INFO  Deployed user-service v2.4.1 (commit: a3f8b2c)
2025-11-15 09:00:05 [user-service] INFO  Starting with config: db_pool_timeout=5000ms, max_retries=3
2025-11-15 12:00:00 [monitoring] INFO  Daily traffic ramp: user-service requests at 2.1x baseline
2025-11-15 13:00:00 [monitoring] INFO  Daily traffic peak: user-service requests at 3.5x baseline
2025-11-15 14:00:00 [monitoring] WARN  Elevated latency: db-primary avg response time 850ms (baseline: 120ms)
2025-11-15 14:15:00 [db-primary] WARN  Table user_sessions: 4.2M rows, last vacuum: 7 days ago
2025-11-15 14:28:00 [cron-scheduler] WARN  Nightly session cleanup was rescheduled from 02:00 to 14:30 (config change in v2.4.1)
""".strip()

# Service dependency map
DEPENDENCY_MAP = {
    "frontend": {"depends_on": ["api-gateway"], "type": "web"},
    "api-gateway": {"depends_on": ["user-service", "order-service", "cache-service"], "type": "proxy"},
    "user-service": {"depends_on": ["db-primary", "cache-service"], "type": "microservice"},
    "order-service": {"depends_on": ["db-primary", "user-service"], "type": "microservice"},
    "cache-service": {"depends_on": ["redis-cluster"], "type": "cache"},
    "db-primary": {"depends_on": [], "type": "database"},
    "redis-cluster": {"depends_on": [], "type": "cache"},
    "load-balancer": {"depends_on": ["user-service", "order-service"], "type": "infra"},
    "cron-scheduler": {"depends_on": ["db-primary"], "type": "scheduler"},
    "monitoring": {"depends_on": [], "type": "observability"},
}

print(f"Incident logs: {len(INCIDENT_LOGS.splitlines())} lines")
print(f"Context logs: {len(CONTEXT_LOGS.splitlines())} lines")
print(f"Services in dependency map: {len(DEPENDENCY_MAP)}")

## 2 - Define Tools

In [ ]:
state = {
    "parsed_events": [],
    "errors": [],
    "hypotheses": [],
    "evidence": [],
}


def parse_log_file(log_text: str, format: str = "standard") -> str:
    """Parse log text into structured events with timestamp, service, level, and message."""
    events = []
    pattern = r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) \[(\S+)\] (\S+)\s+(.*)"
    for line in log_text.strip().splitlines():
        m = re.match(pattern, line.strip())
        if m:
            events.append({
                "timestamp": m.group(1),
                "service": m.group(2),
                "level": m.group(3),
                "message": m.group(4),
            })
    state["parsed_events"].extend(events)
    return json.dumps({
        "parsed_count": len(events),
        "services": list(set(e["service"] for e in events)),
        "levels": {level: sum(1 for e in events if e["level"] == level)
                   for level in set(e["level"] for e in events)},
        "time_range": f"{events[0]['timestamp']} to {events[-1]['timestamp']}" if events else "empty",
    })


def extract_errors(parsed_logs: str = "all") -> str:
    """Extract ERROR and ALERT level events from parsed logs."""
    errors = [e for e in state["parsed_events"] if e["level"] in ("ERROR", "ALERT")]
    state["errors"] = errors

    # Group by service
    by_service = {}
    for e in errors:
        by_service.setdefault(e["service"], []).append({
            "timestamp": e["timestamp"],
            "message": e["message"],
        })
    return json.dumps({
        "total_errors": len(errors),
        "by_service": {svc: {"count": len(evts), "events": evts}
                       for svc, evts in by_service.items()},
    })


def correlate_timestamps(events: list, window_seconds: int = 5) -> str:
    """Find events that occurred within a time window of each other."""
    # Use all parsed events if none specified
    all_events = state["parsed_events"]
    clusters = []
    used = set()

    for i, e1 in enumerate(all_events):
        if i in used:
            continue
        t1 = datetime.strptime(e1["timestamp"], "%Y-%m-%d %H:%M:%S")
        cluster = [e1]
        used.add(i)
        for j, e2 in enumerate(all_events):
            if j in used or j == i:
                continue
            t2 = datetime.strptime(e2["timestamp"], "%Y-%m-%d %H:%M:%S")
            if abs((t2 - t1).total_seconds()) <= window_seconds:
                cluster.append(e2)
                used.add(j)
        if len(cluster) > 1:
            clusters.append({
                "window_start": cluster[0]["timestamp"],
                "event_count": len(cluster),
                "services": list(set(e["service"] for e in cluster)),
                "events": cluster,
            })

    clusters.sort(key=lambda c: c["event_count"], reverse=True)
    return json.dumps({
        "window_seconds": window_seconds,
        "clusters_found": len(clusters),
        "top_clusters": clusters[:5],
    })


def check_dependency(service_name: str) -> str:
    """Look up the dependency chain for a service."""
    info = DEPENDENCY_MAP.get(service_name)
    if not info:
        return json.dumps({"error": f"Unknown service: {service_name}",
                           "available": list(DEPENDENCY_MAP.keys())})

    # Find what depends ON this service (reverse deps)
    dependents = [svc for svc, dep in DEPENDENCY_MAP.items()
                  if service_name in dep["depends_on"]]

    return json.dumps({
        "service": service_name,
        "type": info["type"],
        "depends_on": info["depends_on"],
        "depended_on_by": dependents,
        "impact_radius": len(dependents),
    })


def form_hypothesis(evidence: str) -> str:
    """Record a hypothesis about the root cause along with supporting evidence."""
    hyp = {
        "id": f"H-{len(state['hypotheses'])+1:02d}",
        "evidence": evidence[:500],
        "status": "proposed",
    }
    state["hypotheses"].append(hyp)
    return json.dumps({
        "hypothesis_id": hyp["id"],
        "instruction": (
            "Now test this hypothesis by checking additional logs or dependencies. "
            "Update status to 'confirmed', 'refined', or 'rejected'."
        )
    })


def test_hypothesis(hypothesis_id: str, additional_evidence: str) -> str:
    """Test a hypothesis against additional evidence and update its status."""
    hyp = None
    for h in state["hypotheses"]:
        if h["id"] == hypothesis_id:
            hyp = h
            break
    if not hyp:
        return json.dumps({"error": f"Hypothesis {hypothesis_id} not found"})

    state["evidence"].append({
        "hypothesis_id": hypothesis_id,
        "evidence": additional_evidence,
    })

    return json.dumps({
        "hypothesis_id": hypothesis_id,
        "original_evidence": hyp["evidence"][:200],
        "new_evidence": additional_evidence[:200],
        "instruction": (
            "Based on all evidence, declare the hypothesis 'confirmed', 'refined', or 'rejected'. "
            "If confirmed, write the final incident report. If refined, form a new hypothesis."
        )
    })

## 3 - Tool Dispatcher & Schemas

In [ ]:
TOOL_MAP = {
    "parse_log_file": parse_log_file,
    "extract_errors": extract_errors,
    "correlate_timestamps": correlate_timestamps,
    "check_dependency": check_dependency,
    "form_hypothesis": form_hypothesis,
    "test_hypothesis": test_hypothesis,
}


def dispatch_tool(name: str, arguments: dict) -> str:
    fn = TOOL_MAP.get(name)
    if fn is None:
        return json.dumps({"error": f"Unknown tool: {name}"})
    try:
        return fn(**arguments)
    except Exception as e:
        return json.dumps({"error": str(e)})


tools = [
    {
        "type": "function",
        "function": {
            "name": "parse_log_file",
            "description": "Parse log text into structured events. Returns service list, level counts, and time range.",
            "parameters": {
                "type": "object",
                "properties": {
                    "log_text": {"type": "string", "description": "Raw log text to parse"},
                    "format": {"type": "string", "enum": ["standard", "json", "syslog"], "description": "Log format"}
                },
                "required": ["log_text"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "extract_errors",
            "description": "Extract ERROR and ALERT level events from all parsed logs, grouped by service.",
            "parameters": {
                "type": "object",
                "properties": {
                    "parsed_logs": {"type": "string", "description": "Filter key or 'all' for all parsed logs"}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "correlate_timestamps",
            "description": "Find clusters of events that occurred within a time window of each other.",
            "parameters": {
                "type": "object",
                "properties": {
                    "events": {"type": "array", "items": {"type": "string"}, "description": "Event IDs to correlate (or empty for all)"},
                    "window_seconds": {"type": "integer", "description": "Time window in seconds for correlation"}
                },
                "required": ["window_seconds"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_dependency",
            "description": "Look up the dependency chain for a service. Shows what it depends on and what depends on it.",
            "parameters": {
                "type": "object",
                "properties": {
                    "service_name": {"type": "string", "description": "Service name to look up"}
                },
                "required": ["service_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "form_hypothesis",
            "description": "Record a root cause hypothesis with supporting evidence.",
            "parameters": {
                "type": "object",
                "properties": {
                    "evidence": {"type": "string", "description": "Summary of evidence supporting this hypothesis"}
                },
                "required": ["evidence"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "test_hypothesis",
            "description": "Test a hypothesis against additional evidence.",
            "parameters": {
                "type": "object",
                "properties": {
                    "hypothesis_id": {"type": "string", "description": "Hypothesis ID (e.g., 'H-01')"},
                    "additional_evidence": {"type": "string", "description": "New evidence to test against"}
                },
                "required": ["hypothesis_id", "additional_evidence"]
            }
        }
    }
]

## 4 - Agent Loop

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""\
You are an Incident Root Cause Analysis Agent. Your job is to analyze logs from a
production incident and determine the root cause.

You have two sets of logs:
1. INCIDENT_LOGS: Events during the active incident
2. CONTEXT_LOGS: Events from earlier in the day that may provide context

Workflow:
1. Parse both log files to understand the timeline.
2. Extract all errors and alerts.
3. Correlate events by timestamp to find causal chains.
4. Check service dependencies to understand blast radius.
5. Form a hypothesis about the root cause.
6. Test the hypothesis against available evidence.
7. Refine or confirm, then write the final incident report.

Your final report must include:
- Incident timeline
- Root cause
- Contributing factors
- Impact assessment
- Remediation recommendations

End with DONE when complete.
""")


def run_agent(max_turns: int = 25):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"We had a P1 incident today. Here are the logs:\n\n"
            f"=== INCIDENT LOGS ===\n{INCIDENT_LOGS}\n\n"
            f"=== CONTEXT LOGS (earlier today) ===\n{CONTEXT_LOGS}\n\n"
            f"Please analyze and determine the root cause."
        )}
    ]

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        msg = response.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print(f"\n[Turn {turn+1}] Agent says:\n{msg.content[:800]}")
            if msg.content and "DONE" in msg.content.upper():
                break
            continue

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            # Truncate log_text display
            display_args = {k: (v[:60] + "..." if isinstance(v, str) and len(v) > 60 else v)
                           for k, v in args.items()}
            print(f"[Turn {turn+1}] Tool: {tc.function.name}({json.dumps(display_args)[:120]})")
            result = dispatch_tool(tc.function.name, args)
            print(f"         Result: {result[:200]}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result,
            })

    return messages

In [ ]:
conversation = run_agent(max_turns=25)

## 5 - Results Analysis

In [ ]:
print(f"Events parsed: {len(state['parsed_events'])}")
print(f"Errors extracted: {len(state['errors'])}")
print(f"Hypotheses formed: {len(state['hypotheses'])}")
print(f"Evidence items: {len(state['evidence'])}")

print("\nHypotheses:")
for h in state["hypotheses"]:
    print(f"  [{h['id']}] {h['status']}: {h['evidence'][:100]}...")

In [ ]:
# Build an incident timeline visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

if state["parsed_events"]:
    events_df = pd.DataFrame(state["parsed_events"])
    events_df["timestamp"] = pd.to_datetime(events_df["timestamp"])

    # Only show incident window
    incident_mask = events_df["timestamp"] >= "2025-11-15 14:30:00"
    incident_events = events_df[incident_mask].copy()

    if not incident_events.empty:
        level_colors = {"INFO": "green", "WARN": "orange", "ERROR": "red", "ALERT": "darkred"}
        services = incident_events["service"].unique()
        service_y = {svc: i for i, svc in enumerate(services)}

        fig, ax = plt.subplots(figsize=(14, 6))
        for _, row in incident_events.iterrows():
            ax.scatter(
                row["timestamp"], service_y[row["service"]],
                c=level_colors.get(row["level"], "gray"),
                s=80, alpha=0.7, zorder=5
            )

        ax.set_yticks(range(len(services)))
        ax.set_yticklabels(services)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
        ax.set_xlabel("Time")
        ax.set_title("Incident Timeline by Service")
        ax.grid(axis="x", alpha=0.3)

        # Legend
        for level, color in level_colors.items():
            ax.scatter([], [], c=color, label=level, s=60)
        ax.legend(loc="upper right")

        plt.tight_layout()
        plt.show()

In [ ]:
# Error distribution by service
if state["errors"]:
    errors_df = pd.DataFrame(state["errors"])
    fig, ax = plt.subplots(figsize=(8, 4))
    errors_df["service"].value_counts().plot(kind="barh", ax=ax, color="#d32f2f")
    ax.set_title("Error Count by Service")
    ax.set_xlabel("Error Count")
    plt.tight_layout()
    plt.show()

In [ ]:
# Extract and display the final incident report
for m in reversed(conversation):
    content = m.content if hasattr(m, 'content') else m.get('content')
    if content and "DONE" in (content or "").upper():
        print("=" * 60)
        print("INCIDENT REPORT")
        print("=" * 60)
        print(content)
        break

## 6 - Key Takeaways

1. **Hypothesis-driven analysis**: The agent did not just list errors -- it formed theories and tested them.
2. **Temporal reasoning**: Correlating timestamps revealed the causal chain (slow query -> pool exhaustion -> cascading timeouts).
3. **Dependency awareness**: Checking service dependencies helped the agent understand why order-service failed (it depended on user-service).
4. **Context integration**: The earlier-day logs revealed the real root cause: a deployment changed the cron schedule, causing a massive DELETE during peak traffic.
5. **Structured output**: The final report follows incident management best practices (timeline, root cause, impact, remediation).